# ShopNow Fraud Leaderboard

ShopNow wants to catch fraudulent orders before they ship. For every order of the last two years we know what the shop saw at checkout (order value, device, hour, new customer, ...) and, from the chargeback history, whether it turned out to be **fraud**. That is a classification problem: inputs in, a label out, trained with the same loop as the regression line in the lecture.

**The competition.** Every group trains a neural network (`MLPClassifier` from scikit-learn) on the same training data and reports its **accuracy on the test set**. Highest test accuracy wins. On a tie, the smaller network wins.

**How to work.**
1. Run cells 1 to 3 as they are (data, split, baseline). Do not change them.
2. In cell 4, enter your group name and change the four parameters. Run it. Read the test accuracy.
3. Repeat step 2 as often as you like. Write your best test accuracy on the board, with the parameters.

You have 40 minutes. Using an LLM is allowed, but the parameters are only four, and trying is faster than asking.

**Rules.** The split stays as it is. The model is trained on the training set only. What counts is the number the cell prints for the test set.

In [ ]:
# Cell 1: imports and the ShopNow orders (synthetic, generated with a fixed seed so everyone has the same data)
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(2026)
n = 4000

order_value = np.round(rng.lognormal(mean=4.0, sigma=0.8, size=n), 2)
items = rng.integers(1, 8, n)
new_customer = rng.binomial(1, 0.35, n)
account_age_days = np.where(new_customer == 1, rng.integers(0, 30, n), rng.integers(30, 2000, n))
failed_payments = rng.poisson(0.3, n)
hour = rng.integers(0, 24, n)
device = rng.choice(["desktop", "mobile", "tablet", "tor"], size=n, p=[0.45, 0.42, 0.10, 0.03])
shipping_differs = rng.binomial(1, 0.2, n)

# the hidden truth: how fraud really depends on the inputs (a model has to find this from examples)
night = ((hour >= 0) & (hour < 6)).astype(int)
big_order = (order_value > 150).astype(int)
risk = -1.5
risk += 0.8 * new_customer + 1.0 * failed_payments + 2.5 * (device == "tor") + 0.6 * shipping_differs + 0.5 * night
risk += 2.5 * new_customer * big_order            # a new customer with a big order
risk += 2.0 * night * shipping_differs            # a night order to a different address
risk += 1.5 * (failed_payments >= 2) * big_order  # repeated failed payments on a big order
risk += -1.5 * (account_age_days > 365)           # old accounts are mostly safe
risk += rng.normal(0, 0.3, n)
fraud = rng.binomial(1, 1 / (1 + np.exp(-4 * risk)))

orders = pd.DataFrame({
    "order_value": order_value, "items": items, "new_customer": new_customer,
    "account_age_days": account_age_days, "failed_payments": failed_payments, "hour": hour,
    "device": device, "shipping_differs_from_billing": shipping_differs,
    "fraud": fraud,  # the label: 1 if a chargeback arrived within 90 days
})

print(f"{len(orders)} orders, {orders['fraud'].mean():.1%} fraud")
orders.head(10)

In [ ]:
# Cell 2: train/test split and preprocessing (scaling for numbers, one-hot encoding for the device)
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier

numeric_features = ["order_value", "items", "new_customer", "account_age_days",
                    "failed_payments", "hour", "shipping_differs_from_billing"]
categorical_features = ["device"]
features = numeric_features + categorical_features

train, test = train_test_split(orders, test_size=0.3, random_state=42, stratify=orders["fraud"])
X_train, y_train = train[features], train["fraud"]
X_test, y_test = test[features], test["fraud"]

preprocessing = ColumnTransformer([
    ("numbers", StandardScaler(), numeric_features),
    ("device", OneHotEncoder(), categorical_features),
])

print(f"training set: {len(train)} orders, test set: {len(test)} orders (the model never sees these while learning)")

In [ ]:
# Cell 3: the baseline, a "model" that always says "not fraud"
baseline = 1 - y_test.mean()
print(f"baseline accuracy on the test set: {baseline:.1%}")
print("anything below this number has learned nothing.")

## Cell 4: your model

Four parameters, you know three of them from the regression demo:

| parameter | what it is | try |
|---|---|---|
| `hidden_layer_sizes` | the network: one number per hidden layer, the number is how many neurons. `(8,)` is one layer with 8 neurons, `(32, 16)` is two layers | from `(2,)` to `(256, 256, 256)` |
| `learning_rate_init` | the learning rate: the step size of every update | `0.0001` to `0.5` |
| `max_iter` | how many passes over the training data (epochs) before training stops | `10` to `1000` |
| `alpha` | regularization: a penalty on big weights, keeps the network from memorizing | `0.0001` (almost none) to `1` |

The cell prints the accuracy on the training set **and** on the test set. Only the test set counts. Watch what happens to the gap between the two when the network gets big.

In [ ]:
GROUP = "group 1"

hidden_layer_sizes = (2,)
learning_rate_init = 0.001
max_iter = 20
alpha = 0.0001

model = make_pipeline(
    preprocessing,
    MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, learning_rate_init=learning_rate_init,
                  max_iter=max_iter, alpha=alpha, random_state=0),
)
model.fit(X_train, y_train)

train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)
n_weights = sum(w.size for w in model[-1].coefs_) + sum(b.size for b in model[-1].intercepts_)

print(f"{GROUP}: layers {hidden_layer_sizes}, learning rate {learning_rate_init}, {max_iter} epochs, alpha {alpha}")
print(f"  {n_weights} weights, trained for {model[-1].n_iter_} epochs")
print(f"  training accuracy {train_accuracy:.1%}")
print(f"  TEST accuracy     {test_accuracy:.1%}   (baseline {baseline:.1%})   <- this goes on the board")

## Leaderboard

| group | layers | learning rate | epochs | alpha | training acc. | test acc. |
|---|---|---|---|---|---|---|
| | | | | | | |

**What the numbers mean.** Training accuracy is how well the model fits the examples it learned from. Test accuracy is how well it predicts orders it has never seen, and that is the only number ShopNow cares about. A model that scores 98 % on training and 89 % on test has partly memorized the training data: overfitting.

## After the competition: why the biggest network did not win

Run the cell below. It trains networks of growing size with the same settings and prints training and test accuracy side by side.

In [ ]:
rows = []
for layers in [(2,), (8,), (32,), (32, 16), (64, 64), (256, 256, 256)]:
    m = make_pipeline(preprocessing, MLPClassifier(hidden_layer_sizes=layers, learning_rate_init=0.01,
                                                    max_iter=500, alpha=0.0001, random_state=0))
    m.fit(X_train, y_train)
    rows.append({"layers": str(layers),
                 "weights": sum(w.size for w in m[-1].coefs_) + sum(b.size for b in m[-1].intercepts_),
                 "training accuracy": round(m.score(X_train, y_train), 3),
                 "test accuracy": round(m.score(X_test, y_test), 3)})

pd.DataFrame(rows)

**Three things to take away.**

- More weights always fit the training data better. They do not always predict better: the gap between the two columns is the memorized noise.
- That is why the test set exists: it is the only honest number. If you tune on the test set long enough, it stops being honest too (that is what a validation set is for).
- The lecture's four facts still hold here: the loss is "how surprised was the model by the true label", gradient descent moves the weights, the learning rate is the step size, training stops after `max_iter` epochs or when the loss stops improving.